In [1]:
import pandas as pd
# import numpy as np
# from sklearn.utils import shuffle

# import matplotlib.pyplot as plt
# import pandas as pd
# import seaborn as sns
# import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

# from sklearn.preprocessing import label_binarize
# from sklearn.metrics import roc_curve, auc
from sklearn.utils import shuffle

# from sklearn.multiclass import OneVsRestClassifier
# from sklearn import svm, datasets

from tensorflow.keras import regularizers

# from keras_tuner import RandomSearch
import keras_tuner as kt

# import json
import copy
import gc
# import random

2025-10-31 01:43:41.128777: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-31 01:43:41.156218: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-31 01:43:41.741244: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
print(tf.__version__)

2.20.0


In [3]:
gpus = tf.config.list_physical_devices('GPU')
gpus


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'),
 PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

In [4]:
tf.config.experimental.set_visible_devices(gpus[1], 'GPU')

In [5]:
# from IPython.core.display import display, HTML
# display(HTML("<style>.container { width:99% !important; }</style>"))

In [6]:
# Dc=pd.read_csv('./data/cg_338377.csv')
# Dp=pd.read_csv('./data/py_32319.csv')

# Dc=pd.read_csv('./data/cg_384615.csv')
# Dc=pd.read_csv('./data/cg_train_2019_to_2020-09_119124.csv')
# Dc=pd.read_csv('./data/cg_train_2019_to_2021-09_171515.csv')
Dc=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')

/tmp/ipykernel_225148/3753005390.py:7: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  Dc=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')


In [7]:
lst=[['male', 'female','zy','mz','age','yer','mth','wk','wbc','neu','lym','mon','eos', 'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct', 'plt', 'pdw', 'pct', 'plcr', 'label'],
    ['male', 'female','zy', 'mz', 'age', 'wbc', 'neu', 'lym', 'mon','eos', 'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct','plt','label'],
    ['age', 'wbc', 'neu', 'lym', 'mon','eos', 'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct','plt','label'],
    ['age', 'wbc', 'neu', 'lym', 'mon','eos', 'rbc', 'hgb', 'mcv', 'hct','plt','label']]
fnams=['all','tim_plt3','tim_plt3_sex_zy_mz','tim_plt3_sex_zy_mz_sd_cv']

In [8]:
'''
2019-01 to 2024-09: 384615
2019-01 to 2024-09: 329692
2019-01 to 2023-09: 273708
2019-01 to 2022-09: 228433
2019-01 to 2021-09: 171515
2019-01 to 2020-09: 119124
'''
def get_time_n_data(time_th,D):
    D_list_ten=[]
    for k in range(num):
        # data=pd.read_csv("data/time_fold/cg_times_kfold338377_"+str(k)+".csv",index_col=0)
        # data=pd.read_csv("data/time_fold/cg_times_kfold384615_"+str(k)+".csv",index_col=0) 
        # data=pd.read_csv("data/time_fold/cg_train_2019_to_2020-09_times_kfold119124_"+str(k)+".csv",index_col=0) 
        # data=pd.read_csv("data/time_fold/cg_train_2019_to_2021-09_times_kfold171515_"+str(k)+".csv",index_col=0) 
        D_list_ten.append(data)      
        
    D_num_temp=copy.deepcopy(D_list_ten)  
    D_stats = D.describe()
    D_stats = D_stats.transpose()
    for i in range(num): 
        D_num_temp[i]= (D_num_temp[i] - D_stats['mean']) / (D_stats['std'])
        # D_num_temp[i]= (D_num_temp[i] - D_stats['min']) / (D_stats['max']-D_stats['min'])
        D_num_temp[i].label=D_list_ten[i].label
    return D_num_temp,D_list_ten

In [9]:
# def build_model_base(hp):    
#     model = keras.Sequential()
#     model.add(keras.layers.Flatten(input_shape=[len(x_train.keys())]))
#     for i in range(hp.Int("num_layers", 5, 15)):
#         model.add(layers.Dropout(rate=0.25))
#         model.add(
#                 layers.Dense(
#                     units=hp.Int("units_" + str(i), min_value=10, max_value=200, step=3),
#                     activation="relu",kernel_regularizer=regularizers.l2(0.001)
#                 )
#             )
          
    # model.add(layers.BatchNormalization())
    # model.add(layers.Dense(2))
    # model.compile(
    # optimizer=keras.optimizers.Adam(hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4, 1e-5])),
    # loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    # metrics=["accuracy"],
    # )
    # return model

In [11]:
#para2
def build_model_base(hp):    
    model = keras.Sequential()
    model.add(keras.layers.Flatten(input_shape=[len(x_train.keys())]))
    for i in range(hp.Int("num_layers", 5, 15)):        
        model.add(
                layers.Dense(
                    units=hp.Int("units_" + str(i), min_value=10, max_value=200, step=3),
                    activation="relu",kernel_regularizer=regularizers.l2(0.0001)
                )
            )
          
    # model.add(layers.Dropout(rate=0.25))
        model.add(layers.Dropout(rate=0.15))
    model.add(layers.Dense(2))
    model.compile(
    optimizer=keras.optimizers.Adam(hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4, 1e-5])),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
    )
    return model

In [12]:
#para2
# def build_model_base(hp):    
#     model = keras.Sequential()
#     model.add(keras.layers.Flatten(input_shape=[len(x_train.keys())]))
#     for i in range(hp.Int("num_layers", 5, 15)):        
#         model.add(
#                 layers.Dense(
#                     units=hp.Int("units_" + str(i), min_value=10, max_value=200, step=3),
#                     activation="relu",kernel_regularizer=regularizers.l2(0.0001)
#                 )
#             )
          
#     # model.add(layers.Dropout(rate=0.25))
#     model.add(layers.Dense(2))
#     model.compile(
#     optimizer=keras.optimizers.Adam(hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4, 1e-5])),
#     loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#     metrics=["accuracy"],
#     )
#     return model

In [13]:
# Hyperband: A Novel Bandit-Based Approach to Hyperparameter Optimization
# models={'model_list':[],'history_list':[],'evaluate_list':[]}
def model_fit(x_train,y_train):    
    tuner= kt.Hyperband(build_model_base,
                         objective='val_accuracy',
                         max_epochs=10,
                         factor=3,
                         directory='my_dir',
                         project_name='intro_to_kt',
                         overwrite = True
                       )

    stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15)
    tuner.search(x_train,y_train, epochs=100, validation_split=0.2, shuffle=True, batch_size=512,callbacks=[stop_early])    
    best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

#     使用从搜索中获得的超参数找到训练模型的最佳周期数
    # model_Hyperband= tuner.hypermodel.build(best_hps)
    # history= model_Hyperband.fit(x_train,y_train, epochs=100, validation_split=0.2,  shuffle=True,batch_size=512,callbacks=[stop_early])
    # val_acc_per_epoch =history.history['val_accuracy']
    # best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1

# 重新实例化超模型并使用上面的最佳周期数对其进行训练。    
    model= tuner.hypermodel.build(best_hps)    
    history=model.fit(x_train,y_train, epochs=1000, validation_split=0.2,  shuffle=True,batch_size=512,callbacks=[stop_early])
    # model.save("./models/model_time"+str(time_n)+'_k'+str(fold_n)+'_'+fnam)
    return model

In [13]:
time_n=0
num=10
# bagging_n=60

for i in range(1):
    D_num_temp,_=get_time_n_data(i,Dc)
    # D_num_temp,_=get_time_n_data(i,Dp)
    for n in range(1,2):
        for k in range(10):
            D_train=pd.DataFrame()
            for j in range(num):
                if k!=j:
                    D_train=pd.concat([D_train,D_num_temp[j][lst[n]]],axis=0)    
            D_train=shuffle(D_train)
            x_train,y_train=D_train.drop('label',axis=1),D_train['label']
            model=model_fit(x_train,y_train)
            model.save("./models/model_para2_384615_"+fnams[n]+'_time'+str(i)+'_k'+str(k)+'.keras')
            # model.save("./models/model_para1_119124_"+fnams[n]+'_time'+str(i)+'_k'+str(k)+'.keras')
            # model.save("./models/model_para1_171515_"+fnams[n]+'_time'+str(i)+'_k'+str(k)+'.keras')
            
            keras.backend.clear_session()
            gc.collect()
            # %reset -f
            # del model

Trial 30 Complete [00h 00m 34s]
val_accuracy: 0.7806618213653564

Best val_accuracy So Far: 0.784792959690094
Total elapsed time: 00h 09m 49s
Epoch 1/1000


/home/ddh/.local/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


541/541 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7641 - loss: 0.5176 - val_accuracy: 0.7759 - val_loss: 0.4995
Epoch 2/1000
541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 823us/step - accuracy: 0.7769 - loss: 0.4957 - val_accuracy: 0.7770 - val_loss: 0.4923
Epoch 3/1000
541/541 ━━━━━━━━━━━━━━━━━━━━ 1s 913us/step - accuracy: 0.7780 - loss: 0.4903 - val_accuracy: 0.7801 - val_loss: 0.4862
Epoch 4/1000
541/541 ━━━━━━━━━━━━━━━━━━━━ 1s 983us/step - accuracy: 0.7796 - loss: 0.4868 - val_accuracy: 0.7805 - val_loss: 0.4855
Epoch 5/1000
541/541 ━━━━━━━━━━━━━━━━━━━━ 1s 951us/step - accuracy: 0.7804 - loss: 0.4844 - val_accuracy: 0.7800 - val_loss: 0.4854
Epoch 6/1000
541/541 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7801 - loss: 0.4822 - val_accuracy: 0.7819 - val_loss: 0.4822
Epoch 7/1000
541/541 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7811 - loss: 0.4804 - val_accuracy: 0.7820 - val_loss: 0.4810
Epoch 8/1000
541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 820us/step - accuracy: 0.7817 - loss: 0.4795 - val

In [14]:
x_train

,age,wbc,neu,lym,mon,eos,rbc,hgb,mcv,hct,plt
354628,0.562818,-0.781092,-0.857596,0.054697,-0.462043,-0.027621,0.645585,0.536541,-0.066829,0.706565,-0.480784
19001,-1.572445,-1.035888,-1.079929,-0.080328,-0.413698,-0.247653,0.901351,0.536541,-0.781987,0.513980,-0.252890
186164,0.372169,-0.128044,-0.195769,0.103797,-0.002763,0.001717,2.308066,0.223908,-2.365552,0.770760,0.088950
394392,-0.771721,-0.973795,-0.852426,-0.497680,-0.244489,-0.262322,0.261936,0.625865,0.316291,0.546078,-0.446600
13072,-0.847981,0.332301,0.494495,-0.252179,0.238963,-0.218315,0.389819,-0.222712,-0.960777,-0.095871,-0.537757
...,...,...,...,...,...,...,...,...,...,...,...
246556,0.029002,0.859022,0.931405,0.085385,0.480690,-0.174309,-0.121714,-0.446021,-0.449950,-0.304505,1.057496
87262,0.524688,-0.609801,-0.707651,0.171310,-0.377439,-0.086296,-0.403057,0.000598,0.571705,-0.127969,0.476368
33344,1.477930,-0.866738,-0.780038,-0.381067,-0.389525,-0.188978,-0.121714,0.447217,0.571705,0.225103,-0.457995
272898,0.982244,-0.926690,-0.860182,-0.295142,-0.449956,-0.188978,0.095687,-0.178050,-0.066829,0.112762,-0.366837


In [15]:
## train329692

354628    0
19001     0
186164    1
394392    0
13072     1
         ..
246556    1
87262     0
33344     1
272898    0
326467    1
Name: label, Length: 304540, dtype: int64

In [14]:
Dc=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')
def get_data(Dc):
    D_stats = Dc.describe()
    D_stats = D_stats.transpose()
    D_num_temp= (Dc - D_stats['mean']) / (D_stats['std'])
    D_num_temp.label=Dc.label
    return D_num_temp

/tmp/ipykernel_225148/887575972.py:1: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  Dc=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')


In [16]:
D_num_temp=get_data(Dc)

In [17]:
num=10
for n in range(1,2):
    for k in range(10):
        D_train=D_num_temp[lst[n]]
        D_train=shuffle(D_train)
        x_train,y_train=D_train.drop('label',axis=1),D_train['label']
        model=model_fit(x_train,y_train)
        # model.save("./models/model_para2_train329692_"+fnams[n]+'_k'+str(k)+'.keras')
        model.save("./models/model_para2_drop0.15_train329692_"+fnams[n]+'_k'+str(k)+'.keras')
        keras.backend.clear_session()
        gc.collect()

Trial 30 Complete [00h 00m 34s]
val_accuracy: 0.7565325498580933

Best val_accuracy So Far: 0.7806760668754578
Total elapsed time: 00h 08m 57s
Epoch 1/1000


/home/ddh/.local/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


516/516 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7424 - loss: 0.5653 - val_accuracy: 0.7696 - val_loss: 0.5172
Epoch 2/1000
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7658 - loss: 0.5218 - val_accuracy: 0.7717 - val_loss: 0.5054
Epoch 3/1000
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7688 - loss: 0.5103 - val_accuracy: 0.7781 - val_loss: 0.4968
Epoch 4/1000
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7713 - loss: 0.5030 - val_accuracy: 0.7782 - val_loss: 0.4906
Epoch 5/1000
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7727 - loss: 0.4986 - val_accuracy: 0.7810 - val_loss: 0.4890
Epoch 6/1000
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7726 - loss: 0.4957 - val_accuracy: 0.7806 - val_loss: 0.4817
Epoch 7/1000
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7743 - loss: 0.4933 - val_accuracy: 0.7787 - val_loss: 0.4825
Epoch 8/1000
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7740 - loss: 0.4918 - val_accuracy: